In [3]:
import os
import re
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# 1. Leitura dos documentos do corpora
dir_corpora = "./corpora"
documentos = {}
for i in range(1, 11):
    nome_arq = f"relatorio_{i:02d}.txt"
    with open(os.path.join(dir_corpora, nome_arq), "r", encoding="utf-8") as f:
        documentos[nome_arq] = f.read().strip()

def tokenizar(texto):
    return re.findall(r'\b\w+\b', texto.lower())

corpus_tokenizado = [tokenizar(doc) for doc in documentos.values()]

print(corpus_tokenizado)

# 2. Testando o BM25 com diferentes valores de b (comprimento)
bm25_padrao = BM25Okapi(corpus_tokenizado, k1=1.5, b=0.75)
bm25_sem_len = BM25Okapi(corpus_tokenizado, k1=1.5, b=0.0)

consulta = "rescisão antecipada de contrato e suporte"
q_tokens = tokenizar(consulta)

scores_padrao = bm25_padrao.get_scores(q_tokens)
scores_sem_len = bm25_sem_len.get_scores(q_tokens)

df_comp = pd.DataFrame({
    'Documento': list(documentos.keys()),
    'Score_BM25 (b=0.75)': scores_padrao,
    'Score_BM25 (b=0.0)': scores_sem_len
}).sort_values(by='Score_BM25 (b=0.75)', ascending=False)

print("=== COMPARATIVO DE SENSIBILIDADE DO COMPRIMENTO (BM25) ===")
print(df_comp.head(5))

# 3. LSA / TruncatedSVD para captura de sinonímia
vec = TfidfVectorizer()
X_tfidf = vec.fit_transform(list(documentos.values()))

svd = TruncatedSVD(n_components=3, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)

q_lsa = svd.transform(vec.transform([consulta]))
scores_lsa = cosine_similarity(q_lsa, X_lsa).flatten()

df_lsa = pd.DataFrame({'Documento': list(documentos.keys()), 'Score_LSA': scores_lsa}).sort_values(by='Score_LSA', ascending=False)
print("\n=== SERP LSA (SVD Semântico) ===")
print(df_lsa.head(5))


[['relatório', 'de', 'rescisão', 'contratual', 'de', 'prestação', 'de', 'serviços', 'de', 'ti'], ['manual', 'de', 'suporte', 'e', 'renovação', 'de', 'licença', 'de', 'software', 'corporativo'], ['parecer', 'sobre', 'cláusula', 'de', 'rescisão', 'antecipada', 'e', 'multa', 'contratual'], ['notificação', 'de', 'cobrança', 'de', 'fatura', 'de', 'serviço', 'de', 'telecomunicação'], ['contrato', 'de', 'prestação', 'de', 'serviços', 'com', 'cláusula', 'de', 'suporte'], ['relatório', 'de', 'rescisão', 'de', 'contrato', 'de', 'trabalho', 'por', 'justa', 'causa'], ['termo', 'de', 'confidencialidade', 'e', 'proteção', 'de', 'propriedade', 'intelectual'], ['parecer', 'sobre', 'violação', 'de', 'sigilo', 'e', 'aplicação', 'de', 'multa', 'rescisória'], ['notificação', 'de', 'inadimplência', 'e', 'suspensão', 'de', 'licença', 'de', 'software'], ['auditoria', 'de', 'contrato', 'de', 'prestação', 'de', 'serviços', 'de', 'nuvem', 'e', 'suporte']]
=== COMPARATIVO DE SENSIBILIDADE DO COMPRIMENTO (BM25) =